# 標準住宅の外皮性能

床断熱・基礎断熱の等級4～7について、面積と層構成から UA・ηAC・ηAH を計算します。気象とソルバーは不要です。窓の日射熱取得率、等級4の開口部U値、土間の外周長・線熱貫流率は設計資料から記入してください。

In [ ]:
%pip -q install git+https://github.com/iguchi-lab/vtsimnx.git@75577c91b0bebc54bec54028083fecf9fb310748

In [ ]:
import json
from pathlib import Path
import pandas as pd
import vtsimnx as vt

model = "foundation5"  # 床断熱は default, grade5～grade7。基礎断熱は foundation4～foundation7。
region = 6
house = vt.standard_house
types, layers = house.surface_types[model], house.layers[model]
input_data = {"surfaces": [
    {**row, "layers": layers.get(types[row["surface"]].get("layer_key"))}
    for row in house.surface_rows[model]
]}
# 室温計算用の input_data を組み立てた場合は、その辞書を直接使用できる。
path = Path(f"{model}_region{region}_rating.json")
if not path.exists():
    assessment = house.rating_template(model, region=region)
    path.write_text(json.dumps(assessment, ensure_ascii=False, indent=2), encoding="utf-8")
print(path.resolve())

左側のファイル一覧から表示されたJSONを開き、`null` に設計値を入力して保存します。窓の `eta_d` は枠込みの垂直面日射熱取得率（0～1）です。`lines` には隣接条件ごとの土間外周長 [m]、ψ [W/(m·K)]、温度差係数 H を記入します。編集済みJSONはダウンロードして保存できます。

In [ ]:
assessment = json.loads(path.read_text(encoding="utf-8"))
result = vt.envelope_performance(input_data, assessment)
display(pd.DataFrame([{k: v for k, v in result.items() if k != "parts"}]))
display(pd.DataFrame(result["parts"]))